In [44]:

def get_sample_indices(df):
    i = 0
    new_df = {'pos':{}, 'neg':{}}
    cases = ['pos', 'neg']
    for case in cases:
        for ID in df[case].keys():
            if ID not in new_df[case].keys():
                new_df[case][ID] = {}
            sample_count = int(df[case][ID])
            new_df[case][ID]['sample_count'] = sample_count
            start = i
            end = sample_count+i-1
            sample_indices = [start, end] if start!=end else [start]
            i += sample_count
            new_df[case][ID]['sample_indices'] = sample_indices
    return new_df

In [45]:
import json
from utils.data_loader import load_dict

In [46]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

# ── 1. Load JAADbeh test data ──────────────────────────────────────────────
root = 'preprocessed_data/jaadbeh_with_c_g/test/'  # adjust path to your jaadbeh folder
x_c_test = np.load(root + 'x_c.npy')  # (N, 16, 2)
x_g_test = np.load(root + 'x_g.npy')  # (N, 2)
y_test   = np.load(root + 'y.npy')    # (N, 1)

import json
with open(root + 'ID2sample.json') as f:
    ped_dict = json.load(f)
ped_dict = get_sample_indices(ped_dict)

# ── 2. Extract one sample per pedestrian (last window) ────────────────────
last_indices = []
labels = []

for label, group in [(1, 'pos'), (0, 'neg')]:
    for ped_id, info in ped_dict[group].items():
        last_idx = info['sample_indices'][-1]  # handles both [start,end] and [single]
        last_indices.append(last_idx)
        labels.append(label)

last_indices = np.array(last_indices)
labels = np.array(labels)

# sanity check
print(f"Total pedestrians: {len(last_indices)}")
print(f"Crossers (pos): {labels.sum()}")
print(f"Non-crossers (neg): {(labels==0).sum()}")
print(f"Max index: {last_indices.max()}, Test set size: {len(x_c_test)}")

# ── 3. Extract per-pedestrian features ────────────────────────────────────
x_c_ped = x_c_test[last_indices]   # (n_peds, 16, 2)
x_g_ped = x_g_test[last_indices]   # (n_peds, 2)

# last frame of the 16-frame window for time-series features
x_c_last = x_c_ped[:, -1, :]       # (n_peds, 2)

# confirmed from preprocessing code:
# x_c[:, 0] = tra_light (traffic signal)
# x_c[:, 1] = veh_speed (vehicle speed)
# x_g[:, 0] = road_type
# x_g[:, 1] = intersection

# ── 4. Merge sparse categories ────────────────────────────────────────────
traffic = x_c_last[:, 0].copy()
traffic[traffic == 2.0] = 1.0      # merge unknown into not-green

speed = x_c_last[:, 1].copy()
speed[speed == 1.0] = 2.0          # merge moving_slow into moving_fast

# ── 5. Chi-square + permutation test ──────────────────────────────────────
def cramers_v(chi2, n, k, r):
    return np.sqrt(chi2 / (n * (min(k, r) - 1)))

def permutation_chi2(feat, labels, n_permutations=10000):
    feat = np.array(feat)
    ct = pd.crosstab(feat, labels)
    observed_chi2, _, _, _ = chi2_contingency(ct)
    count = sum(
        chi2_contingency(pd.crosstab(feat, np.random.permutation(labels)))[0] >= observed_chi2
        for _ in range(n_permutations)
    )
    return observed_chi2, count / n_permutations

features = {
    'traffic_signal': traffic,
    'vehicle_speed':  speed,
    'road_type':      x_g_ped[:, 0],
    'intersection':   x_g_ped[:, 1],
}

print("\n" + "="*65)
for name, feat in features.items():
    ct = pd.crosstab(feat, labels)
    chi2, p, dof, expected = chi2_contingency(ct)
    min_exp = expected.min()
    n = ct.values.sum()
    k, r = ct.shape
    cv = cramers_v(chi2, n, k, r)
    
    print(f"\n{name} contingency table:")
    print(ct)
    print(f"Min expected frequency: {min_exp:.2f}")
    
    if min_exp < 5:
        # use permutation test instead
        chi2_stat, p_perm = permutation_chi2(feat, labels)
        print(f"→ Permutation test: chi2={chi2_stat:.2f}, p={p_perm:.4f}, CramerV={cv:.3f}")
    else:
        print(f"→ Chi-square: chi2={chi2:.2f}, p={p:.4f}, CramerV={cv:.3f}")

Total pedestrians: 200
Crossers (pos): 115
Non-crossers (neg): 85
Max index: 18789, Test set size: 18790


traffic_signal contingency table:
col_0   0    1
row_0         
0.0    82  109
1.0     3    6
Min expected frequency: 3.83
→ Permutation test: chi2=0.05, p=0.7438, CramerV=0.016

vehicle_speed contingency table:
col_0   0   1
row_0        
0.0     6  29
3.0    33  61
4.0    46  25
Min expected frequency: 14.88
→ Chi-square: chi2=25.75, p=0.0000, CramerV=0.359

road_type contingency table:
col_0   0    1
row_0         
0      62  103
1      23   12
Min expected frequency: 14.88
→ Chi-square: chi2=8.24, p=0.0041, CramerV=0.203

intersection contingency table:
col_0   0    1
row_0         
0      40    7
1      45  108
Min expected frequency: 19.98
→ Chi-square: chi2=43.39, p=0.0000, CramerV=0.466
